In [92]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression,Ridge,Lasso,SGDRegressor,RANSACRegressor
from sklearn.tree import DecisionTreeRegressor # Corrected import for DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score


print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"sklearn version: {sklearn.__version__}")

pandas version: 2.2.2
numpy version: 2.0.2
sklearn version: 1.6.1


In [20]:
df = pd.read_csv("/content/drive/MyDrive/Synoris Training/Day6_training/Housing.csv")
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [22]:
df.shape

(545, 13)

In [8]:
df.nunique()

,0
price,219
area,284
bedrooms,6
bathrooms,4
stories,4
mainroad,2
guestroom,2
basement,2
hotwaterheating,2
airconditioning,2


# now we are going to apply encoding on that which have object type and categorical form

In [11]:
encoder = LabelEncoder()

In [15]:
label_encod = []
for col in df.columns:
  if df[col].nunique() == 2:
    label_encod.append(col)
label_encod

['mainroad',
 'guestroom',
 'basement',
 'hotwaterheating',
 'airconditioning',
 'prefarea']

In [17]:
for col in label_encod:
  df[col] = encoder.fit_transform(df[col])
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,furnished
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,furnished
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,semi-furnished
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,furnished
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,furnished


In [18]:
new_df = df.copy()
new_df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,furnished
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,furnished
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,semi-furnished
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,furnished
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,furnished


In [23]:
new_df.shape

(545, 13)

In [28]:
# Create a OneHotEncoder instance
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the 'furnishingstatus' column
furnishing_encoded = ohe.fit_transform(new_df[['furnishingstatus']])

# Create a DataFrame from the encoded data
furnishing_df = pd.DataFrame(furnishing_encoded, columns=ohe.get_feature_names_out(['furnishingstatus']), index=new_df.index)

# Drop the original 'furnishingstatus' column from new_df
new_df = new_df.drop('furnishingstatus', axis=1)

# Concatenate the original DataFrame with the one-hot encoded DataFrame
new_df = pd.concat([new_df, furnishing_df], axis=1)

# Now calculate the correlation
new_df.corr()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_furnished,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
price,1.000000,0.535997,0.366494,0.517545,0.420712,0.296898,0.255517,0.187057,0.093073,0.452954,0.384394,0.329777,0.229350,0.063656,-0.280587
area,0.535997,1.000000,0.151858,0.193820,0.083996,0.288874,0.140297,0.047417,-0.009229,0.222393,0.352980,0.234779,0.145772,0.006156,-0.142278
bedrooms,0.366494,0.151858,1.000000,0.373930,0.408564,-0.012033,0.080549,0.097312,0.046049,0.160603,0.139270,0.079023,0.079054,0.050040,-0.126252
bathrooms,0.517545,0.193820,0.373930,1.000000,0.326165,0.042398,0.126469,0.102106,0.067159,0.186915,0.177496,0.063472,0.108139,0.029834,-0.132107
stories,0.420712,0.083996,0.408564,0.326165,1.000000,0.121706,0.043538,-0.172394,0.018847,0.293602,0.045547,0.044425,0.093176,-0.003648,-0.082972
mainroad,0.296898,0.288874,-0.012033,0.042398,0.121706,1.000000,0.092337,0.044002,-0.011781,0.105423,0.204433,0.199876,0.129971,0.011450,-0.133123
guestroom,0.255517,0.140297,0.080549,0.126469,0.043538,0.092337,1.000000,0.372066,-0.010308,0.138179,0.037466,0.160897,0.099721,0.005821,-0.099023
basement,0.187057,0.047417,0.097312,0.102106,-0.172394,0.044002,0.372066,1.000000,0.004385,0.047341,0.051497,0.228083,0.069852,0.050284,-0.117935
hotwaterheating,0.093073,-0.009229,0.046049,0.067159,0.018847,-0.011781,-0.010308,0.004385,1.000000,-0.130023,0.067864,-0.059411,-0.008472,0.063819,-0.059194
airconditioning,0.452954,0.222393,0.160603,0.186915,0.293602,0.105423,0.138179,0.047341,-0.130023,1.000000,0.159173,0.117382,0.160994,-0.053179,-0.094086


In [29]:
new_df.shape

(545, 15)

In [32]:
X = new_df.drop('price', axis=1)
y = new_df['price']

In [33]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [34]:
X_train

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_furnished,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
46,6000,3,2,4,1,0,0,0,1,1,0,1.0,0.0,0.0
93,7200,3,2,1,1,0,1,0,1,3,0,0.0,1.0,0.0
335,3816,2,1,1,1,0,1,0,1,2,0,1.0,0.0,0.0
412,2610,3,1,2,1,0,1,0,0,0,1,0.0,0.0,1.0
471,3750,3,1,2,1,0,0,0,0,0,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,6000,4,2,4,1,0,0,0,1,0,0,0.0,0.0,1.0
106,5450,4,2,1,1,0,1,0,1,0,1,0.0,1.0,0.0
270,4500,3,2,3,1,0,0,1,0,1,0,1.0,0.0,0.0
435,4040,2,1,1,1,0,0,0,0,0,0,0.0,0.0,1.0


In [35]:
y_train

,price
46,7525000
93,6300000
335,3920000
412,3430000
471,3010000
...,...
71,6755000
106,6160000
270,4340000
435,3290000


In [47]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Linear Regression model


In [48]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled,y_train)

LinearRegression()

In [49]:
y_train_pred = lr_model.predict(X_train_scaled)
y_test_pred = lr_model.predict(X_test_scaled)

In [50]:
print(f"mean squared error for train data: {sklearn.metrics.mean_squared_error(y_train,y_train_pred)}")
print(f"mean squared error for test data: {sklearn.metrics.mean_squared_error(y_test,y_test_pred)}")

print(f"mean absolute error for train data: {sklearn.metrics.mean_absolute_error(y_train,y_train_pred)}")
print(f"mean absolute error for test data: {sklearn.metrics.mean_absolute_error(y_test,y_test_pred)}")

mean squared error for train data: 968358188440.7242
mean squared error for test data: 1754318687330.6677
mean absolute error for train data: 719242.8936724715
mean absolute error for test data: 970043.4039201643


# Lasso

In [52]:
print(f"R2 score for Lasso: {r2_score(y_test, y_test_pred)}")

R2 score for Lasso: 0.6529242642153177


In [53]:
Lasso_model = Lasso()
Lasso_model.fit(X_train_scaled,y_train)

Lasso()

In [54]:
y_train_pred = Lasso_model.predict(X_train_scaled)
y_test_pred = Lasso_model.predict(X_test_scaled)



In [55]:
print(f"mean square error of lasso: {mean_squared_error(y_test,y_test_pred)}")
print(f"mean absolute error of lasso: {mean_absolute_error(y_test,y_test_pred)}")

mean square error of lasso: 1754320649501.432
mean absolute error of lasso: 970043.8988344392


In [56]:
print(f"R2_score for the lasso {r2_score(y_test,y_test_pred)}")

R2_score for the lasso 0.6529238760179699


# Ridge

In [57]:
Ridge_mode = Ridge()

In [59]:
Ridge_mode.fit(X_train_scaled,y_train)

Ridge()

In [61]:
y_train_pred = Ridge_mode.predict(X_train_scaled)
y_test_pred = Ridge_mode.predict(X_test_scaled)

In [62]:
print(f"mean square error of ridge: {mean_squared_error(y_test,y_test_pred)}")
print(f"mean absolute error of ridge: {mean_absolute_error(y_test,y_test_pred)}")

mean square error of ridge: 1754931410491.0757
mean absolute error of ridge: 969943.1930536417


In [64]:
print(f"R2_score for the ridge {r2_score(y_test,y_test_pred)}")

R2_score for the ridge 0.6528030426018977


In [65]:
import numpy as np

mse_ridge = mean_squared_error(y_test, y_test_pred)
rmse_ridge = np.sqrt(mse_ridge)

# Define a validation baseline for RMSE (example value)
# You can adjust this baseline as needed based on business requirements or domain knowledge.
validation_baseline_rmse = 1000000  # Example baseline

print(f"Ridge Model RMSE: {rmse_ridge}")
print(f"Validation Baseline RMSE: {validation_baseline_rmse}")

if rmse_ridge < validation_baseline_rmse:
    print("Success: Ridge Model RMSE is below the validation baseline.")
else:
    print("Failure: Ridge Model RMSE is NOT below the validation baseline.")

Ridge Model RMSE: 1324738.2422543238
Validation Baseline RMSE: 1000000
Failure: Ridge Model RMSE is NOT below the validation baseline.


# XGBRegression


In [84]:
XGB = SGDRegressor()

In [85]:
XGB.fit(X_train_scaled,y_train)

SGDRegressor()

In [86]:
y_pred =  XGB.predict(X_test_scaled)

In [88]:
print(f"mean square error of ridge: {mean_squared_error(y_test,y_pred)}")
print(f"mean absolute error of ridge: {mean_absolute_error(y_test,y_pred)}")

mean square error of ridge: 1758083210040.7373
mean absolute error of ridge: 968729.2720990207


In [90]:
print(f"R2_score for the ridge {r2_score(y_test,y_pred)}")

R2_score for the ridge 0.6521794881954808


# RandomForestRegressor

In [93]:
RDR = RandomForestRegressor()

In [94]:
RDR.fit(X_train_scaled,y_train)

RandomForestRegressor()

In [95]:
y_pred = RDR.predict(X_test_scaled)

In [96]:
print(f"mean square error of ridge: {mean_squared_error(y_test,y_pred)}")
print(f"mean absolute error of ridge: {mean_absolute_error(y_test,y_pred)}")

mean square error of ridge: 1946751744482.6326
mean absolute error of ridge: 1023508.3308868501


In [97]:
print(f"r2_score: {r2_score(y_test,y_pred)}")

r2_score: 0.6148531626631029


In [75]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

In [76]:
rf = RandomForestRegressor(random_state=42)

In [77]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

In [78]:
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='r2')
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42),
             param_grid={'max_depth': [5, 10, None],
                         'min_samples_split': [2, 5],
                         'n_estimators': [50, 100, 200]},
             scoring='r2')

In [79]:
y_pred = grid_search.predict(X_test_scaled)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [80]:
print(f"mean square error of ridge: {mean_squared_error(y_test,y_pred)}")
print(f"mean absolute error of ridge: {mean_absolute_error(y_test,y_pred)}")

mean square error of ridge: 6312257698418.544
mean absolute error of ridge: 1842216.2486738802


In [81]:
print(f"r2_score of random forest {r2_score(y_test,y_pred)}")

r2_score of random forest -0.24882183662687996


# Prediction by Neural Networks


In [67]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

In [69]:
model = Sequential([
    Input((14,)), # Changed from 12 to 14 to match the number of features in X_train

    Dense(8055, activation='relu'),
    Dense(5035, activation='relu'),
    Dense(515, activation='relu'),
    Dense(1, activation='linear')
])

model.compile(loss='mean_absolute_error', optimizer='adam')
model.fit(X_train, y_train,epochs=50)

Epoch 1/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 92ms/step - loss: 4310294.5000
Epoch 2/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1918014.6250
Epoch 3/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1423307.7500
Epoch 4/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1336469.7500
Epoch 5/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1364314.6250
Epoch 6/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1371520.6250
Epoch 7/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1328619.6250
Epoch 8/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1336448.0000
Epoch 9/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1299771.0000
Epoch 10/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1301286.5000
Epoch 11/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1313924.0000
Epoch 12/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1307071.5000
Epoch 13/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1314338.2500
Epoch 14/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 

In [82]:
import joblib

# Define the filename for the saved model
model_filename = 'linear_regression_model.joblib'

# Save the Linear Regression model
joblib.dump(lr_model, model_filename)
print(f"Model saved successfully as {model_filename}")

# You can also load the model back to verify
loaded_model = joblib.load(model_filename)
print("Model loaded successfully.")

# Verify by making a prediction (optional)
# For example, predict on the first test sample
sample_prediction = loaded_model.predict(X_test_scaled[0].reshape(1, -1))
print(f"Prediction with loaded model for first test sample: {sample_prediction[0]}")

Model saved successfully as linear_regression_model.joblib
Model loaded successfully.
Prediction with loaded model for first test sample: 5164653.900339669
